In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np

In [2]:
# CONSTANTS:
# ALPHAS = [0.01, 0.025, 0.05, 0.1, 0.25, 0.5]
ALPHAS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

# I should probably export the max_ind_set indices into a file so the selection is fixed and reproducible
# We should also fix our phenotype manifest so our arbitrary index is fixed

In [3]:
phenotype_manifest = pd.read_csv("phenotype_manifest.csv", usecols=[
    'description',
    'in_max_independent_set',
    'phenocode',
    'filename'
    ])

In [4]:
phenotype_manifest['description'] = phenotype_manifest['description'].fillna(phenotype_manifest['phenocode'])
max_ind_set = phenotype_manifest[phenotype_manifest['in_max_independent_set']]

In [5]:
exclude_phenotypes = []

In [6]:
with open("exclude_phenotypes.txt", "r") as file:
    for line in file:
        if line[0]=='#':
            continue
        exclude_phenotypes.append(int(line.strip().split()[0]))

In [7]:
expected_folders = []

for index, row in max_ind_set.iterrows():
    if index in exclude_phenotypes:
        expected_folders.append(np.nan)
    else:
        expected_folders.append(os.path.splitext(os.path.basename(row['filename']))[0])

In [8]:
metadata_columns = [
    "index",
    "description",
    "alpha",
    "dir_name",
    "num_snps_found",
    "num_coding_snps_found",
    "num_overlapping_snps",
    "num_overlapping_loci",
    "num_original_list",
    "num_original_coding_snps",
    "p_value_threshold",
    "percentage_loci_recovered",
    "ld_based_clumping",
    "r2_threshold",
    "kb_radius",
    "window_size"]
collected_metadata_df = pd.DataFrame(columns=metadata_columns)
collected_metadata_df['description'] = max_ind_set['description']
collected_metadata_df['index'] = max_ind_set.index
collected_metadata_df['dir_name'] = expected_folders
collected_metadata_df = collected_metadata_df.loc[collected_metadata_df.index.repeat(len(ALPHAS))].reset_index(drop=True)
collected_metadata_df['alpha'] = ALPHAS * len(max_ind_set)

In [14]:
collected_metadata_df

,index,description,alpha,dir_name,num_snps_found,num_coding_snps_found,num_overlapping_snps,num_overlapping_loci,num_original_list,num_original_coding_snps,p_value_threshold,percentage_loci_recovered,ld_based_clumping,r2_threshold,kb_radius,window_size
0,0,Albumin,0.5,biomarkers-30600-both_sexes-irnt.tsv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,Albumin,1.0,biomarkers-30600-both_sexes-irnt.tsv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,Albumin,1.5,biomarkers-30600-both_sexes-irnt.tsv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,Albumin,2.0,biomarkers-30600-both_sexes-irnt.tsv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,Albumin,2.5,biomarkers-30600-both_sexes-irnt.tsv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,7043,latanoprost,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
896,7043,latanoprost,1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
897,7043,latanoprost,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
898,7043,latanoprost,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
missing_indeces = []

for idx, row in collected_metadata_df.iterrows():
    alpha_folder = f"deepcast_sd_ablation/results_{str(row['alpha'])}"
    folder_path = f"{alpha_folder}/{row['dir_name']}"
    if not Path(folder_path):
        raise ValueError("No phecode directories found in 'patho_phenotypes/deepcast_phenotypes' folder")
    metadata_path = f"{folder_path}/metadata_df.csv"
    try:
        # Read the small single-row CSV
        temp_df = pd.read_csv(metadata_path)
        
        # Now update the corresponding fields in df1
        for col in temp_df.columns:
            if col in collected_metadata_df.columns:
                collected_metadata_df.at[idx, col] = temp_df.iloc[0][col]
                
    except FileNotFoundError:
        print(f"File {metadata_path} not found, skipping.")
        missing_indeces.append((idx, row['description'], row['alpha']))    

File deepcast_sd_ablation/results_0.5/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_1.0/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_1.5/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_2.0/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_2.5/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_3.0/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_0.5/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_1.0/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_1.5/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_2.0/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_2.5/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_3.0/nan/metadata_df.csv not found, skipping.
File deepcast_sd_ablation/results_0.5/nan/metadata_d

In [22]:
# excluded phenotypes
collected_metadata_df['num_snps_found'].isna().sum()

np.int64(85)

In [23]:
# timed out phenotypes
collected_metadata_df['dir_name'].isna().sum()

np.int64(84)

In [19]:
collected_metadata_df

,index,description,alpha,dir_name,num_snps_found,num_coding_snps_found,num_overlapping_snps,num_overlapping_loci,num_original_list,num_original_coding_snps,p_value_threshold,percentage_loci_recovered,ld_based_clumping,r2_threshold,kb_radius,window_size
0,0,Albumin,0.5,biomarkers-30600-both_sexes-irnt.tsv,870,173,419,949,975,103,0.0,0.973333,True,0.2,500,NaN
1,0,Albumin,1.0,biomarkers-30600-both_sexes-irnt.tsv,774,241,252,925,975,103,0.0,0.948718,True,0.2,500,NaN
2,0,Albumin,1.5,biomarkers-30600-both_sexes-irnt.tsv,712,289,186,907,975,103,0.0,0.930256,True,0.2,500,NaN
3,0,Albumin,2.0,biomarkers-30600-both_sexes-irnt.tsv,662,321,160,891,975,103,0.0,0.913846,True,0.2,500,NaN
4,0,Albumin,2.5,biomarkers-30600-both_sexes-irnt.tsv,626,352,137,872,975,103,0.000001,0.894359,True,0.2,500,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,7043,latanoprost,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
896,7043,latanoprost,1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
897,7043,latanoprost,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
898,7043,latanoprost,2.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
missing_indeces

[(150, 'Cereal type', 0.5),
 (151, 'Cereal type', 1.0),
 (152, 'Cereal type', 1.5),
 (153, 'Cereal type', 2.0),
 (154, 'Cereal type', 2.5),
 (155, 'Cereal type', 3.0),
 (294, 'Hearing difficulty/problems with background noise', 0.5),
 (295, 'Hearing difficulty/problems with background noise', 1.0),
 (296, 'Hearing difficulty/problems with background noise', 1.5),
 (297, 'Hearing difficulty/problems with background noise', 2.0),
 (298, 'Hearing difficulty/problems with background noise', 2.5),
 (299, 'Hearing difficulty/problems with background noise', 3.0),
 (324, 'Qualifications', 0.5),
 (325, 'Qualifications', 1.0),
 (326, 'Qualifications', 1.5),
 (327, 'Qualifications', 2.0),
 (328, 'Qualifications', 2.5),
 (329, 'Qualifications', 3.0),
 (330, 'Qualifications', 0.5),
 (331, 'Qualifications', 1.0),
 (332, 'Qualifications', 1.5),
 (333, 'Qualifications', 2.0),
 (334, 'Qualifications', 2.5),
 (335, 'Qualifications', 3.0),
 (336, 'Current employment status', 0.5),
 (337, 'Current employ

In [28]:
collected_metadata_df.to_csv("collected_metadata.csv")